# TIỀN XỬ LÝ & LÀM SẠCH DỮ LIỆU CẢNH BÁO HỌC TẬP (`Tienxl.ipynb`)

Quy trình thực hiện:
1. **Đọc 2 file CSV gốc** từ thư mục `datasets/raw`:
   - `DS_HOCTIEP_da_gan_nhan_hoc_luc_canh_bao.csv`
   - `DS_HOCTIEP_GOP_2_FILE_DA_CHUAN_HOA.csv`
2. **Làm sạch & Chuẩn hóa dữ liệu**:
   - Bỏ 2 dòng tiêu đề thừa ở đầu file (`header=2`), chuẩn hóa tên cột.
   - Đồng nhất cấu trúc cột giữa 2 file (`Ngành`, `Khóa`, `Kỳ xét`, `TBC HK`, `TBC TL`, `Số TC Đạt`, `Số TC F`, `Học lực`, `Cảnh báo`), loại bỏ cột `Lớp` bị khuyết >84% dữ liệu.
   - Chuẩn hóa chuỗi ký tự và đồng nhất tên ngành bị lệch chữ hoa/thường (`Công nghệ kỹ thuật Cơ khí` -> `Công nghệ kỹ thuật cơ khí`).
   - Ép kiểu dữ liệu số, loại bỏ các dòng `NaN`, kiểm tra miền giá trị hợp lệ (`0 <= TBC HK, TBC TL <= 4`) và loại bỏ các bản ghi lỗi (toàn bộ chỉ số học tập bằng 0).
   - Gộp 2 file và khử hoàn toàn các bản ghi trùng lặp.
3. **Chia sẵn tập dữ liệu 80% Train – 20% Test (Stratified Split)**:
   - Gắn trực tiếp cột `split` (`'train'` / `'test'`) vào dữ liệu đã làm sạch theo tỷ lệ 80% Train và 20% Test (phân tầng theo `Học lực` và `Cảnh báo`).
4. **Lưu trữ vào `datasets/processed`**:
   - Xuất ra 2 định dạng: **`cleaned_data.parquet`** và **`cleaned_data.json`**.

In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    candidate_roots = ['/content/drive/MyDrive/FMNN_CODE/FMN_EduWarning']
except ImportError:
    candidate_roots = [
        os.path.abspath(os.path.join(os.getcwd(), '..')),
        os.path.abspath(os.path.join(os.getcwd(), '..', 'FMN_EduWarning')),
        os.path.abspath(os.path.join(os.getcwd(), 'FMN_EduWarning')),
        os.path.abspath(os.getcwd())
    ]

project_root = next((p for p in candidate_roots if os.path.exists(os.path.join(p, 'datasets', 'raw'))), candidate_roots[0])
raw_data_path = os.path.join(project_root, 'datasets', 'raw')
processed_data_path = os.path.join(project_root, 'datasets', 'processed')
os.makedirs(processed_data_path, exist_ok=True)

print(f"📂 Thư mục dữ liệu gốc (raw)      : {raw_data_path}")
print(f"📂 Thư mục dữ liệu sạch (processed): {processed_data_path}")

📂 Thư mục dữ liệu gốc (raw)      : d:\NCKH\FMN_EduWarning\datasets\raw
📂 Thư mục dữ liệu sạch (processed): d:\NCKH\FMN_EduWarning\datasets\processed


In [ ]:
# 2. Đọc 2 file CSV từ thư mục datasets/raw (bỏ 2 dòng tiêu đề thừa ở đầu với header=2)
csv_file_1 = os.path.join(raw_data_path, 'DS_HOCTIEP_da_gan_nhan_hoc_luc_canh_bao.csv')
csv_file_2 = os.path.join(raw_data_path, 'DS_HOCTIEP_GOP_2_FILE_DA_CHUAN_HOA.csv')

df_raw_1 = pd.read_csv(csv_file_1, encoding='utf-8-sig', header=2)
df_raw_2 = pd.read_csv(csv_file_2, encoding='utf-8-sig', header=2)

# Xóa khoảng trắng dư thừa ở tên cột
df_raw_1.columns = df_raw_1.columns.str.strip()
df_raw_2.columns = df_raw_2.columns.str.strip()

print(f"✅ Đã đọc File 1 ({os.path.basename(csv_file_1)}): {df_raw_1.shape}")
print(f"   Các cột File 1: {df_raw_1.columns.tolist()}")
print(f"✅ Đã đọc File 2 ({os.path.basename(csv_file_2)})     : {df_raw_2.shape}")
print(f"   Các cột File 2: {df_raw_2.columns.tolist()}")

✅ Đã đọc File 1 (DS_HOCTIEP_da_gan_nhan_hoc_luc_canh_bao.csv): (21878, 10)
   Các cột File 1: ['STT', 'Ngành', 'Khóa', 'Kỳ xét', 'TBC HK', 'TBC TL', 'Số TC Đạt', 'Số TC F', 'Cảnh báo', 'Học lực']
✅ Đã đọc File 2 (DS_HOCTIEP_GOP_2_FILE_DA_CHUAN_HOA.csv)     : (25985, 11)
   Các cột File 2: ['STT', 'Lớp', 'Ngành', 'Khóa', 'Kỳ xét', 'TBC HK', 'TBC TL', 'Số TC Đạt', 'Số TC F', 'Học lực', 'Cảnh báo']


In [ ]:
# 3. Làm sạch và chuẩn hóa dữ liệu từ 2 file CSV
COMMON_COLS = ['Ngành', 'Khóa', 'Kỳ xét', 'TBC HK', 'TBC TL', 'Số TC Đạt', 'Số TC F', 'Học lực', 'Cảnh báo']
FLOAT_COLS = ['TBC HK', 'TBC TL']
INT_COLS = ['Kỳ xét', 'Số TC Đạt', 'Số TC F', 'Học lực', 'Cảnh báo']

# Gộp dữ liệu từ 2 file theo đúng thứ tự cột chuẩn
df_combined = pd.concat([df_raw_2[COMMON_COLS], df_raw_1[COMMON_COLS]], ignore_index=True)
initial_count = len(df_combined)

# 3.1. Chuẩn hóa chuỗi ký tự và đồng nhất tên ngành bị viết lệch hoa/thường
df_combined['Ngành'] = (
    df_combined['Ngành']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .replace({'Công nghệ kỹ thuật Cơ khí': 'Công nghệ kỹ thuật cơ khí'})
)
df_combined['Khóa'] = df_combined['Khóa'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

# 3.2. Ép kiểu dữ liệu số và loại bỏ các dòng chứa NaN
for col in FLOAT_COLS + INT_COLS:
    df_combined[col] = pd.to_numeric(df_combined[col], errors='coerce')

after_nan_df = df_combined.dropna(subset=COMMON_COLS).copy()
nan_removed = initial_count - len(after_nan_df)

for col in INT_COLS:
    after_nan_df[col] = after_nan_df[col].astype(int)

# 3.3. Kiểm tra miền giá trị hợp lệ và loại bỏ các dòng lỗi (toàn bộ chỉ số học tập bằng 0)
valid_mask = (
    (after_nan_df['TBC HK'] >= 0.0) & (after_nan_df['TBC HK'] <= 4.0) &
    (after_nan_df['TBC TL'] >= 0.0) & (after_nan_df['TBC TL'] <= 4.0) &
    (after_nan_df['Số TC Đạt'] >= 0) &
    (after_nan_df['Số TC F'] >= 0) &
    (after_nan_df['Kỳ xét'] >= 1) &
    (after_nan_df['Học lực'].isin([1, 2, 3, 4, 5, 6])) &
    (after_nan_df['Cảnh báo'].isin([0, 1])) &
    ~(
        (after_nan_df['TBC HK'] == 0.0) &
        (after_nan_df['TBC TL'] == 0.0) &
        (after_nan_df['Số TC Đạt'] == 0) &
        (after_nan_df['Số TC F'] == 0)
    )
)
valid_df = after_nan_df[valid_mask].copy()
invalid_removed = len(after_nan_df) - len(valid_df)

# 3.4. Khử trùng lặp dữ liệu giữa 2 file và đánh lại STT
df_cleaned = valid_df.drop_duplicates(subset=COMMON_COLS).reset_index(drop=True)
duplicates_removed = len(valid_df) - len(df_cleaned)
df_cleaned.insert(0, 'STT', np.arange(1, len(df_cleaned) + 1))

print("=" * 65)
print("          BÁO CÁO QUÁ TRÌNH LÀM SẠCH & GỘP DỮ LIỆU")
print("=" * 65)
print(f"• Tổng số bản ghi ban đầu khi gộp 2 file : {initial_count:,} dòng")
print(f"• Số bản ghi bị thiếu giá trị (NaN)      : {nan_removed:,} dòng")
print(f"• Số bản ghi sai miền giá trị / rỗng 0   : {invalid_removed:,} dòng")
print(f"• Số bản ghi trùng lặp đã loại bỏ        : {duplicates_removed:,} dòng")
print(f"• Tổng số bản ghi sạch còn lại           : {len(df_cleaned):,} dòng")
print(f"• Số lượng ngành học sau chuẩn hóa       : {df_cleaned['Ngành'].nunique()} ngành")
print("=" * 65)

          BÁO CÁO QUÁ TRÌNH LÀM SẠCH & GỘP DỮ LIỆU
• Tổng số bản ghi ban đầu khi gộp 2 file : 47,863 dòng
• Số bản ghi bị thiếu giá trị (NaN)      : 0 dòng
• Số bản ghi sai miền giá trị / rỗng 0   : 4 dòng
• Số bản ghi trùng lặp đã loại bỏ        : 22,896 dòng
• Tổng số bản ghi sạch còn lại           : 24,963 dòng
• Số lượng ngành học sau chuẩn hóa       : 47 ngành


In [ ]:
# 4. Chia sẵn dữ liệu theo tỷ lệ 80% Train và 20% Test (Stratified theo Học lực & Cảnh báo)
stratify_key = df_cleaned['Học lực'].astype(str) + '_' + df_cleaned['Cảnh báo'].astype(str)

train_idx, test_idx = train_test_split(
    df_cleaned.index,
    test_size=0.20,
    random_state=42,
    stratify=stratify_key
)

# Gán cột 'split' ('train' / 'test') trực tiếp vào bảng dữ liệu
df_cleaned['split'] = 'train'
df_cleaned.loc[test_idx, 'split'] = 'test'

n_train = int((df_cleaned['split'] == 'train').sum())
n_test = int((df_cleaned['split'] == 'test').sum())
print("✅ Đã chia dữ liệu thành công:")
print(f"   - Tập Train (80%): {n_train:,} mẫu ({n_train / len(df_cleaned) * 100:.3f}%)")
print(f"   - Tập Test  (20%): {n_test:,} mẫu ({n_test / len(df_cleaned) * 100:.3f}%)")

# 5. Lưu dữ liệu sau khi làm sạch & chia tập ra 2 file Parquet và JSON trong datasets/processed
parquet_output_path = os.path.join(processed_data_path, 'cleaned_data.parquet')
json_output_path = os.path.join(processed_data_path, 'cleaned_data.json')

df_cleaned.to_parquet(parquet_output_path, index=False)
df_cleaned.to_json(json_output_path, orient='records', force_ascii=False, indent=2)

print("\n💾 Đã lưu 2 file dữ liệu sạch (kèm nhãn chia tập 'split') vào thư mục processed:")
print(f"   1. Parquet : {parquet_output_path} ({os.path.getsize(parquet_output_path) / 1024:.2f} KB)")
print(f"   2. JSON    : {json_output_path} ({os.path.getsize(json_output_path) / 1024:.2f} KB)")

✅ Đã chia dữ liệu thành công:
   - Tập Train (80%): 19,970 mẫu (79.998%)
   - Tập Test  (20%): 4,993 mẫu (20.002%)

💾 Đã lưu 2 file dữ liệu sạch (kèm nhãn chia tập 'split') vào thư mục processed:
   1. Parquet : d:\NCKH\FMN_EduWarning\datasets\processed\cleaned_data.parquet (256.77 KB)
   2. JSON    : d:\NCKH\FMN_EduWarning\datasets\processed\cleaned_data.json (6304.89 KB)


In [ ]:
# 6. Hàm nạp tự động Train/Test (80/20) từ file Parquet hoặc JSON cho phần code thuật toán
FEATURE_COLS = ['TBC HK', 'TBC TL', 'Số TC Đạt', 'Số TC F', 'Kỳ xét']

def load_preprocessed_splits(processed_dir=processed_data_path, prefer_format='parquet'):
    """
    Đọc dữ liệu đã làm sạch và chia sẵn 80% Train / 20% Test từ thư mục datasets/processed.
    Hỗ trợ đọc trực tiếp từ cleaned_data.parquet hoặc cleaned_data.json.
    """
    parquet_file = os.path.join(processed_dir, 'cleaned_data.parquet')
    json_file = os.path.join(processed_dir, 'cleaned_data.json')

    if prefer_format == 'parquet' and os.path.exists(parquet_file):
        df = pd.read_parquet(parquet_file)
    elif os.path.exists(json_file):
        df = pd.read_json(json_file, orient='records')
    elif os.path.exists(parquet_file):
        df = pd.read_parquet(parquet_file)
    else:
        raise FileNotFoundError(f"Không tìm thấy cleaned_data.parquet hoặc cleaned_data.json trong {processed_dir}")

    df_train = df[df['split'] == 'train']
    df_test = df[df['split'] == 'test']

    X_train = df_train[FEATURE_COLS].values
    X_test = df_test[FEATURE_COLS].values
    y_hl_train = df_train['Học lực'].values
    y_hl_test = df_test['Học lực'].values
    y_cb_train = df_train['Cảnh báo'].values
    y_cb_test = df_test['Cảnh báo'].values

    return X_train, X_test, y_hl_train, y_hl_test, y_cb_train, y_cb_test

# Chạy kiểm tra nạp thử
X_train, X_test, y_hl_train, y_hl_test, y_cb_train, y_cb_test = load_preprocessed_splits()
print("✅ Kiểm tra nạp tự động tập Train/Test từ file đã xử lý:")
print(f"   • X_train : {X_train.shape} | y_hl_train : {y_hl_train.shape} | y_cb_train : {y_cb_train.shape}")
print(f"   • X_test  : {str(X_test.shape):<10} | y_hl_test  : {str(y_hl_test.shape):<8} | y_cb_test  : {y_cb_test.shape}")

✅ Kiểm tra nạp tự động tập Train/Test từ file đã xử lý:
   • X_train : (19970, 5) | y_hl_train : (19970,) | y_cb_train : (19970,)
   • X_test  : (4993, 5)  | y_hl_test  : (4993,)  | y_cb_test  : (4993,)
